In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("BusinessReportingLab") \
    .master("local[*]") \
    .getOrCreate()
print("Spark Session created successfully.")
spark.sparkContext.setLogLevel("WARN")

In [ ]:
employee_df = spark.read.csv(
    "employee_data.csv",
    header=True,
    inferSchema=True
)

print("EMPLOYEE DATA")
employee_df.show(truncate=False)

print("SCHEMA")
employee_df.printSchema()

In [ ]:
from pyspark.sql.functions import to_date

employee_df = employee_df.withColumn(
    "JoiningDate",
    to_date("JoiningDate", "yyyy-MM-dd")
)

employee_df.printSchema()

In [ ]:
employee_df.createOrReplaceTempView(
    "employees"
)

In [ ]:
spark.sql("""
    SELECT *
    FROM employees
""").show()

In [ ]:
department_count = spark.sql("""
    SELECT
        Department,
        COUNT(*) AS EmployeeCount
    FROM employees
    GROUP BY Department
    ORDER BY EmployeeCount DESC
""")

department_count.show()

In [ ]:
department_salary = spark.sql("""
    SELECT
        Department,
        COUNT(*) AS EmployeeCount,
        SUM(Salary) AS TotalSalary,
        ROUND(AVG(Salary), 2) AS AverageSalary,
        MIN(Salary) AS MinimumSalary,
        MAX(Salary) AS MaximumSalary
    FROM employees
    GROUP BY Department
    ORDER BY TotalSalary DESC
""")

department_salary.show()

In [ ]:
city_department_report = spark.sql("""
    SELECT
        Department,
        City,
        COUNT(*) AS EmployeeCount
    FROM employees
    GROUP BY Department, City
    ORDER BY Department, EmployeeCount DESC
""")

city_department_report.show()

In [ ]:
department_report = spark.sql("""
    SELECT
        Department,
        COUNT(*) AS TotalEmployees,
        SUM(Salary) AS TotalSalaryCost,
        ROUND(AVG(Salary), 2) AS AverageSalary,
        MIN(Salary) AS LowestSalary,
        MAX(Salary) AS HighestSalary
    FROM employees
    GROUP BY Department
    ORDER BY TotalSalaryCost DESC
""")

print("DEPARTMENT-WISE REPORT")

department_report.show()

In [ ]:
salary_summary = spark.sql("""
    SELECT
        COUNT(*) AS TotalEmployees,
        SUM(Salary) AS TotalSalaryCost,
        ROUND(AVG(Salary), 2) AS AverageSalary,
        MIN(Salary) AS LowestSalary,
        MAX(Salary) AS HighestSalary
    FROM employees
""")

salary_summary.show()

In [ ]:
salary_category = spark.sql("""
    SELECT
        EmployeeID,
        Name,
        Department,
        Salary,

        CASE
            WHEN Salary >= 70000 THEN 'High'
            WHEN Salary >= 50000 THEN 'Medium'
            ELSE 'Low'
        END AS SalaryCategory

    FROM employees

    ORDER BY Salary DESC
""")

salary_category.show()

In [ ]:
salary_category.createOrReplaceTempView(
    "salary_categories"
)

In [ ]:
salary_category_report = spark.sql("""
    SELECT
        SalaryCategory,
        COUNT(*) AS EmployeeCount,
        ROUND(AVG(Salary), 2) AS AverageSalary
    FROM salary_categories
    GROUP BY SalaryCategory
    ORDER BY AverageSalary DESC
""")

salary_category_report.show()

In [ ]:
top_employees = spark.sql("""
    SELECT
        EmployeeID,
        Name,
        Department,
        Designation,
        Salary
    FROM employees
    ORDER BY Salary DESC
    LIMIT 5
""")

top_employees.show()

In [ ]:
salary_ranking = spark.sql("""
    SELECT
        EmployeeID,
        Name,
        Department,
        Salary,

        RANK() OVER(
            ORDER BY Salary DESC
        ) AS SalaryRank,
        DENSE_RANK() OVER(
                    ORDER BY Salary DESC
                ) AS DenseSalaryRank

    FROM employees

    ORDER BY SalaryRank
""")

salary_ranking.show()

In [ ]:
department_ranking = spark.sql("""
    SELECT
        EmployeeID,
        Name,
        Department,
        Salary,

        DENSE_RANK() OVER(
            PARTITION BY Department
            ORDER BY Salary DESC
        ) AS DepartmentRank

    FROM employees

    ORDER BY Department, DepartmentRank
""")

department_ranking.show()

In [ ]:
highest_paid_department = spark.sql("""
    SELECT
        Name,
        Department,
        Designation,
        Salary
    FROM
    (
        SELECT
            Name,
            Department,
            Designation,
            Salary,

            DENSE_RANK() OVER(
                PARTITION BY Department
                ORDER BY Salary DESC
            ) AS SalaryRank

        FROM employees
    )

    WHERE SalaryRank = 1

    ORDER BY Salary DESC
""")

highest_paid_department.show()

In [ ]:
salary_contribution = spark.sql("""
    SELECT
        Name,
        Department,
        Salary,

        ROUND(
            Salary * 100.0 /
            SUM(Salary) OVER(),
            2
        ) AS SalaryContributionPercent

    FROM employees

    ORDER BY SalaryContributionPercent DESC
""")

salary_contribution.show()

In [ ]:
trend_df = spark.sql("""
    SELECT
        *,
        YEAR(JoiningDate) AS JoiningYear,
        MONTH(JoiningDate) AS JoiningMonth,
        DATE_FORMAT(JoiningDate, 'yyyy-MM') AS YearMonth
    FROM employees
""")

trend_df.createOrReplaceTempView(
    "employee_trends"
)

trend_df.show()

In [ ]:
monthly_hiring = spark.sql("""
    SELECT
        YearMonth,
        COUNT(*) AS NewEmployees
    FROM employee_trends
    GROUP BY YearMonth
    ORDER BY YearMonth
""")

monthly_hiring.show()

In [ ]:
department_hiring_trend = spark.sql("""
    SELECT
        YearMonth,
        Department,
        COUNT(*) AS NewEmployees
    FROM employee_trends
    GROUP BY YearMonth, Department
    ORDER BY YearMonth, Department
""")

department_hiring_trend.show()

In [ ]:
monthly_salary_trend = spark.sql("""
    SELECT
        YearMonth,
        COUNT(*) AS NewEmployees,
        SUM(Salary) AS SalaryAdded,
        ROUND(AVG(Salary), 2) AS AverageNewHireSalary
    FROM employee_trends
    GROUP BY YearMonth
    ORDER BY YearMonth
""")

monthly_salary_trend.show()

In [ ]:
monthly_hiring.createOrReplaceTempView(
    "monthly_hiring"
)

cumulative_hiring = spark.sql("""
    SELECT
        YearMonth,
        NewEmployees,

        SUM(NewEmployees) OVER(
            ORDER BY YearMonth
            ROWS BETWEEN UNBOUNDED PRECEDING
            AND CURRENT ROW
        ) AS TotalEmployeesHired

    FROM monthly_hiring

    ORDER BY YearMonth
""")

cumulative_hiring.show()

In [ ]:
monthly_salary_trend.createOrReplaceTempView(
    "monthly_salary"
)

In [ ]:
salary_growth = spark.sql("""
    SELECT
        YearMonth,
        SalaryAdded,

        LAG(SalaryAdded) OVER(
            ORDER BY YearMonth
        ) AS PreviousMonthSalary,

        SalaryAdded -
        LAG(SalaryAdded) OVER(
            ORDER BY YearMonth
        ) AS SalaryDifference

    FROM monthly_salary

    ORDER BY YearMonth
""")

salary_growth.show()

In [ ]:
print("===== DEPARTMENT REPORT =====")

department_report.show(
    truncate=False
)

In [ ]:
print("===== SALARY ANALYTICS =====")

department_ranking.show(
    truncate=False
)

In [ ]:
print("===== MONTHLY TREND REPORT =====")

monthly_salary_trend.show(
    truncate=False
)

In [30]:
department_report.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("output/department_report")